# Step 1: Feature Extraction & Dataset Preparation
**PYNQ-Z2 Drone vs Bird Classification**

This notebook:
1. Loads the raw IQ radar dataset
2. Extracts a 64-element feature vector from each 1280-sample IQ segment
3. Groups into 4 super-classes: Drone / Bird / Human / Corner Reflector
4. Splits into train/val/test and saves as `.npy` files

⚠️ Place `dataset,npy` in the same folder as this notebook before running.


In [ ]:
import numpy as np
import os
import time

# === LOAD DATASET ===
DATASET_PATH = 'dataset,npy'  # Note: comma in filename
assert os.path.exists(DATASET_PATH), f"Dataset not found at {DATASET_PATH}!"

print("Loading dataset...")
data = np.load(DATASET_PATH, allow_pickle=True)
print(f"Dataset shape: {data.shape}")  # Should be (130, 6)
print(f"Columns: label, IQ_data, range_m, time_s, split_flag, edge_flag")


In [ ]:
# === EXPLORE THE DATASET ===
from collections import Counter

all_labels = []
total_segments = 0
for row_idx in range(data.shape[0]):
    raw_label = data[row_idx, 0]
    label = str(raw_label[0]).strip() if hasattr(raw_label, "__len__") and not isinstance(raw_label, str) else str(raw_label).strip()
    iq_matrix = data[row_idx, 1]
    n_seg = iq_matrix.shape[1]
    total_segments += n_seg
    all_labels.extend([label] * n_seg)

print(f"Total rows (measurements): {data.shape[0]}")
print(f"Total segments: {total_segments}")
print(f"\nSegments per class:")
for label, count in sorted(Counter(all_labels).items(), key=lambda x: -x[1]):
    print(f"  {label:35s} : {count:6d}")


In [ ]:
# === FEATURE EXTRACTION FUNCTION ===
def extract_features(iq_vector):
    """
    Extract 64 features from a 1280-element complex IQ vector.
    
    Features:
      [0:32]  - Downsampled power spectrum of center range cell
      [32:48] - Cross-range-cell magnitude variance (micro-Doppler indicator)
      [48:63] - Per-range-cell statistics (mean, std, dynamic range)
      [63]    - Mean phase difference (motion indicator)
    
    Returns INT8 values in [0, 255] range.
    """
    # Reshape: 5 range cells x 256 azimuth samples
    iq_2d = iq_vector.reshape(5, 256)
    mag = np.abs(iq_2d)  # (5, 256)
    
    # Feature 1: Power spectrum of center range cell (index 2)
    center_cell = iq_2d[2, :]
    fft_result = np.fft.fft(center_cell)
    power_spectrum = np.abs(fft_result[:128])  # positive freq only
    # Downsample 128 -> 32 bins
    power_32 = power_spectrum.reshape(32, 4).mean(axis=1)
    
    # Feature 2: Magnitude variance across range cells at each azimuth
    mag_var = np.var(mag, axis=0)  # (256,)
    var_16 = mag_var.reshape(16, 16).mean(axis=1)  # downsample to 16
    
    # Feature 3: Per-range-cell statistics
    stats = []
    for i in range(5):
        cell_mag = mag[i, :]
        stats.append(np.mean(cell_mag))
        stats.append(np.std(cell_mag))
        stats.append(np.max(cell_mag) - np.min(cell_mag))
    stats = np.array(stats)  # 15 values
    
    # Feature 4: Mean absolute phase difference (1 value)
    phase_diff = np.mean(np.abs(np.diff(np.angle(center_cell))))
    
    # Concatenate: 32 + 16 + 15 + 1 = 64
    features = np.concatenate([power_32, var_16, stats, [phase_diff]])
    
    # Normalize to [0, 255] for INT8
    fmin = features.min()
    fmax = features.max()
    if fmax - fmin > 1e-10:
        features = (features - fmin) / (fmax - fmin) * 255.0
    else:
        features = np.zeros_like(features)
    
    return features.astype(np.float32)  # keep float32 for training, quantize later


# Quick test
test_iq = data[0, 1][:, 0]
test_feat = extract_features(test_iq)
print(f"Feature vector shape: {test_feat.shape}")
print(f"Feature range: [{test_feat.min():.1f}, {test_feat.max():.1f}]")
print(f"First 10 features: {test_feat[:10]}")


In [ ]:
# === CLASS MAPPING ===
def get_superclass(label):
    """Map 15 fine-grained labels to 4 super-classes."""
    # Handle numpy arrays: ["D1"] -> "D1"
    if hasattr(label, "__len__") and not isinstance(label, str):
        label = str(label[0]).strip()
    else:
        label = str(label).strip()
    if label.startswith('D'):
        return 0  # Drone
    elif label in ['seagull', 'pigeon', 'raven', 'black-headed gull',
                    'seagull and black-headed gull', 'heron']:
        return 1  # Bird
    elif label in ['human_walk', 'human_run']:
        return 2  # Human
    elif label == 'CR':
        return 3  # Corner Reflector
    else:
        print(f"WARNING: Unknown label '{label}'")
        return -1

CLASS_NAMES = ['Drone', 'Bird', 'Human', 'Corner Reflector']
print("Class mapping:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i} = {name}")


In [ ]:
# === BUILD FEATURE DATASET ===
print("Extracting features from all segments...")
start_time = time.time()

X_train, y_train = [], []
X_val, y_val = [], []
X_test, y_test = [], []

skipped = 0
processed = 0

for row_idx in range(data.shape[0]):
    raw_label = data[row_idx, 0]
    label = str(raw_label[0]).strip() if hasattr(raw_label, "__len__") and not isinstance(raw_label, str) else str(raw_label).strip()
    iq_matrix = data[row_idx, 1]       # (1280, N_segments)
    split_flags = data[row_idx, 4]     # 1=train, 2=val, 3=test
    
    superclass = get_superclass(label)
    if superclass == -1:
        skipped += iq_matrix.shape[1]
        continue
    
    n_segments = iq_matrix.shape[1]
    
    for seg_idx in range(n_segments):
        iq_vec = iq_matrix[:, seg_idx]
        features = extract_features(iq_vec)
        
        # Handle split_flags - could be array or scalar
        if hasattr(split_flags, '__len__') and len(split_flags) > seg_idx:
            split = int(split_flags[seg_idx])
        elif hasattr(split_flags, '__len__'):
            split = int(split_flags[0])
        else:
            split = int(split_flags)
        
        if split == 1:
            X_train.append(features)
            y_train.append(superclass)
        elif split == 2:
            X_val.append(features)
            y_val.append(superclass)
        else:
            X_test.append(features)
            y_test.append(superclass)
        
        processed += 1
        if processed % 10000 == 0:
            print(f"  Processed {processed} segments...")

elapsed = time.time() - start_time
print(f"\nDone! Processed {processed} segments in {elapsed:.1f}s (skipped {skipped})")


In [ ]:
# === CONVERT TO NUMPY ARRAYS ===
X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.int64)
X_val = np.array(X_val, dtype=np.float32)
y_val = np.array(y_val, dtype=np.int64)
X_test = np.array(X_test, dtype=np.float32)
y_test = np.array(y_test, dtype=np.int64)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape},   y={y_val.shape}")
print(f"Test:  X={X_test.shape},  y={y_test.shape}")

print(f"\nClass distribution (train):")
for c in range(4):
    count = (y_train == c).sum()
    print(f"  {CLASS_NAMES[c]:20s}: {count:6d} ({100*count/len(y_train):.1f}%)")


In [ ]:
# === SAVE ===
np.save('X_train.npy', X_train)
np.save('y_train.npy', y_train)
np.save('X_val.npy', X_val)
np.save('y_val.npy', y_val)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)

print("Saved: X_train.npy, y_train.npy, X_val.npy, y_val.npy, X_test.npy, y_test.npy")
print("\n✅ Step 1 complete. Proceed to Step 2 (Training).")
